# Multi-agent Collaboration for Financial Analysis

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Using python3.11.3

!pip3 install -r ../requirements.txt

/Users/namitjain/.zshenv:.:1: no such file or directory: /Users/namitjain/.cargo/env

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from crewai import Agent, Task, Crew

In [4]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [5]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [6]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [9]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [23]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}) at the moment."
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [24]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
        "Consider latest news ? ({news_impact_consideration})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)

In [25]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)

In [26]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

In [27]:
from crewai import Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-4o", 
                           temperature=0.5),
    process=Process.hierarchical,
    verbose=True
)

2025-02-11 23:11:08,454 - 8637429824 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [28]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

In [29]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

# Agent: Crew Manager
## Task: Continuously monitor and analyze market data for the selected stock (AAPL) at the moment.Use statistical modeling and machine learning to identify trends and predict market movements.
# Agent: Data Analyst
## Task: Continuously monitor and analyze market data for the selected stock (AAPL) at the moment using statistical modeling and machine learning to identify trends and predict market movements.


# Agent: Data Analyst
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"AAPL stock analysis October 2023\"}"
## Tool Output: 
{'searchParameters': {'q': 'AAPL stock analysis October 2023', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', 'link': 'https://finance.yahoo.com/quote/AAPL/history/', 'snippet': 'Discover historical prices for AAPL stock on Yahoo Finance. View daily, weekly or monthly format back to when Apple Inc. stock was is

In [32]:
from IPython.display import Markdown
Markdown(result.raw)

Given the current context for AAPL trading, here's a thorough analysis of potential risks and mitigation strategies:

1. **Technical Analysis Risks**:
   - **Price Volatility**: The current price is $234.19, with a tight range for entry and exit points. The MACD reading of -2.41 suggests bearish momentum, while the RSI at 41.67 indicates some room for upward movement but not oversold conditions. Technical indicators show mixed signals.
   - **Mitigation**: Setting a stop-loss slightly below $225 is a prudent measure to manage downside risk. It's critical to stay vigilant for significant changes in these indicators and adjust entry/exit points accordingly.

2. **Sentiment and Market Dynamics**:
   - **Catalysts**: The recent partnership with Alibaba is enhancing market sentiment. However, such partnerships can also attract speculative trading, increasing volatility.
   - **Mitigation**: Monitor news closely related to the partnership's progress and any market speculation. Utilize tranche-based allocation to moderate exposure and capitalize on gradual market movements.

3. **Execution Risks**:
   - **Market Timing**: The strategy relies on timing an entry below $235 and limiting risk with a stop-loss. Market fluctuations could result in missing ideal entry or exit points due to rapid sentiment-driven changes.
   - **Mitigation**: Use advanced trading platforms to implement limit orders effectively. Also, consider utilizing short-term options to hedge against abrupt market movements that could adversely affect stock positions.

4. **Volume and Liquidity Risks**:
   - **Volume Spikes**: These could indicate increased interest and potential volatility spikes. While volume is a positive indicator for liquidity, it can make stock movement more complex.
   - **Mitigation**: Close observation of trading volumes is essential. Set alerts for unusual volume spikes to adapt the strategy quickly and maintain liquidity advantage.

Overall, while the partnership with Alibaba introduces a positive catalyst, traders must remain cautious of technical signals and market dynamics. Employing systematic risk management techniques and staying updated with business developments are key to mitigating the risks associated with this trading plan on AAPL.
```